In [1]:
import re
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
from multiprocessing.pool import ThreadPool
from Bio import SeqIO
from Bio.Seq import Seq

def read_fa(path):
    res = {}
    rescords = list(SeqIO.parse(path, format="fasta"))
    for x in rescords:
        id = str(x.id)
        seq = str(x.seq)
        res[id] = seq
    return res

In [2]:
ss_loops_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/binding_features_dic.txt", header=None, index_col=0) #Solo se utiliza para obtener indices
index_ss_lops = ss_loops_dict.index
print(index_ss_lops)

test_lnc = [element.split('_')[0] for element in index_ss_lops]
test_mir = [element.split('_')[1] for element in index_ss_lops]

Index(['NONHSAT000179.2_hsa-miR-25-3p', 'NONHSAT000179.2_hsa-miR-299-3p',
       'NONHSAT000179.2_hsa-miR-92a-3p', 'NONHSAT000179.2_hsa-miR-92b-3p',
       'NONHSAT000201.2_hsa-miR-136-5p', 'NONHSAT000201.2_hsa-miR-148a-3p',
       'NONHSAT000201.2_hsa-miR-148b-3p', 'NONHSAT000201.2_hsa-miR-152-3p',
       'NONHSAT000201.2_hsa-miR-15a-5p', 'NONHSAT000201.2_hsa-miR-15b-5p',
       ...
       'NONHSAT064005.2_hsa-miR-128-3p', 'NONHSAT001974.2_hsa-miR-145-5p',
       'NONHSAT046854.2_hsa-miR-130b-3p', 'NONHSAT040965.2_hsa-miR-302b-3p',
       'NONHSAT004493.2_hsa-miR-339-5p', 'NONHSAT021837.2_hsa-miR-454-3p',
       'NONHSAT097840.2_hsa-miR-20a-5p', 'NONHSAT135445.2_hsa-miR-485-5p',
       'NONHSAT061665.2_hsa-miR-539-5p', 'NONHSAT033924.2_hsa-miR-21-5p'],
      dtype='object', name=0, length=45977)


In [3]:
# Obtener binding info para mir
def delete_no_interact_zone(seq):
    #seq = seq.replace('t', '')
    #seq = seq.replace('a', '')
    #seq = seq.replace('c', '')
    #seq = seq.replace('g', '')
    #seq = seq.replace('u', '')
    seq = seq.replace('-','')
    return seq

def get_binding_loops(binding_info_dic_especifico, lnc, mir):
    multi_dic = {}
    hairpin_dic = {}
    interior_dic = {}
    mir = binding_info_dic_especifico["name_mir_q"]
    binding_path = f"/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA Precursor/ss_loops_mirna/loops_{mir}.txt"
    with open(binding_path, 'r') as archivo:
        lineas = archivo.readlines()
        seq_precursor = lineas[-2].strip()
    seq_especifico = binding_info_dic_especifico["q_seq"] 
    especifico_start = binding_info_dic_especifico["q_start"]  
    especifico_end = binding_info_dic_especifico["q_end"] 
    seq_precursor = Seq(seq_precursor.upper())
    seq_especifico = Seq(seq_especifico.upper())
    start_ubi = seq_precursor.find(seq_especifico) 
    if start_ubi == -1:
        seq_especifico_flipped = seq_especifico[::-1]
        start_ubi = seq_precursor.find(seq_especifico_flipped)
    start_ubi += 1
    fin_ubi = start_ubi + len(seq_especifico) -1
    seq_especifico = seq_especifico[especifico_start -1:especifico_end]
    start = seq_precursor.find(seq_especifico) 
    if start == -1:
        seq_especifico_flipped = seq_especifico[::-1]
        start = seq_precursor.find(seq_especifico_flipped)
    start += 1
    fixed_s_index =  int(start) 
    fixed_e_index =  int(start) + len(seq_especifico) -1
    with open('/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA Precursor/indices_binding_features_miRNA_precursor.txt', 'a') as output_ind:
        output_ind.write(f"{lnc}_{mir}, Inicio Ubicacion mir: {start_ubi}, Fin Ubicacion mir: {fin_ubi}, Inicio Sitio de Union: {fixed_s_index}, Fin Sitio de Union: {fixed_e_index}\n")
    binding_indexes = {*range(fixed_s_index, fixed_e_index + 1)}
    pattern = r'\((.*?)\)'
    values_energy=[]
    with open(binding_path, "r") as file:
        for line in file.readlines():
            limits=[0,0]
            limits_s=[0,0]
            line_segments = line.strip().split()
            ranges = re.findall(pattern, line)
            if "External loop" in line:
                values_energy.append(int(line_segments[-1]))
            if len(line_segments) > 2 and len(ranges) > 0:
                limits = re.findall(pattern, line)[0].strip().split(',')
                try:
                    limits_s = re.findall(pattern, line)[1].strip().split(',')
                except:
                    pass
                values_energy.append(int(line_segments[-1]))
                if (int(limits[0]) in binding_indexes or
                int(limits[1]) in binding_indexes or 
                int(limits_s[0]) in binding_indexes or 
                int(limits_s[1]) in binding_indexes):
                    if "Interior" in line_segments:
                        interior_dic[f'{limits[0]}, {limits[1]}'] = int(line_segments[-1])
                    elif "Hairpin" in line_segments:
                        hairpin_dic[f'{limits[0]}, {limits[1]}'] = int(line_segments[-1])
                    elif "Multi" in line_segments:
                        multi_dic[f'{limits[0]}, {limits[1]}'] = int(line_segments[-1])

        total_energy = np.sum(values_energy)/100

    return interior_dic, hairpin_dic, multi_dic, total_energy

def get_features(interior_dic, hairpin_dic, multi_dic, binding_info):
    interaction_energy = binding_info["align_energy"]
    num_interior = len(interior_dic)
    num_hairpin = len(hairpin_dic)
    num_multi = len(multi_dic)
    binding_zone_energy = sum(interior_dic.values()) + sum(hairpin_dic.values()) + sum(multi_dic.values())
    return [str(num_interior), str(num_hairpin), str(num_multi), str(binding_zone_energy/100), str(interaction_energy), ] 

with open('/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA Precursor/binding_features_miRNA_precursor.txt', 'w') as output:
    tamano_no_hits_found=0
    for i in tqdm(range(len(test_lnc))):
        lnc = test_lnc[i]
        mir = test_mir[i]

        path_precursor = f'/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/features/features mir precursor/feature_{lnc}_{mir}.txt'
        path_especifico = f'/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/features/features mir especifico/feature_{lnc}_{mir}.txt'

        binding_info_dic_precursor = {"name_lnc_r": lnc, "name_mir_q": mir}
        binding_info_dic_especifico = {"name_lnc_r": lnc, "name_mir_q": mir}

        negative = False

        with open(path_precursor, "r") as file:
            for line in file.readlines():
                line_segments = line.strip().split()
                if 'Forward:	Score:' in line:
                    binding_info_dic_precursor["score"] = line_segments[2]
                    binding_info_dic_precursor["q_start"] = int(line_segments[3].split(":")[1])
                    binding_info_dic_precursor["q_end"] = int(line_segments[5])
                    binding_info_dic_precursor["r_start"] = int(line_segments[6].split(":")[1])
                    binding_info_dic_precursor["r_end"] = int(line_segments[8])
                    binding_info_dic_precursor["align_len"] = int(line_segments[11].replace('(', '').replace(')', ''))
                    binding_info_dic_precursor["percent_1"] = line_segments[12].replace('(', '').replace('%)', '')
                    binding_info_dic_precursor["percent_2"] = line_segments[13].replace('(', '').replace('%)', '')
                if 'Query:' in line:
                    q_seq = line_segments[2]
                    binding_info_dic_precursor["q_seq"] = delete_no_interact_zone(q_seq)
                if 'Ref:' in line:
                    r_seq = line_segments[2]
                    binding_info_dic_precursor["r_seq"] = delete_no_interact_zone(r_seq)
                if 'Energy:' in line:
                    binding_info_dic_precursor["align_energy"] = line_segments[1]
                if 'No Hits Found above Threshold' in line:
                    negative = True
                    tamano_no_hits_found+=1

        with open(path_especifico, "r") as file:
            for line in file.readlines():
                line_segments = line.strip().split()
                if 'Forward:	Score:' in line:
                    binding_info_dic_especifico["score"] = line_segments[2]
                    binding_info_dic_especifico["q_start"] = int(line_segments[3].split(":")[1])
                    binding_info_dic_especifico["q_end"] = int(line_segments[5])
                    binding_info_dic_especifico["r_start"] = int(line_segments[6].split(":")[1])
                    binding_info_dic_especifico["r_end"] = int(line_segments[8])
                    binding_info_dic_especifico["align_len"] = int(line_segments[11].replace('(', '').replace(')', ''))
                    binding_info_dic_especifico["percent_1"] = line_segments[12].replace('(', '').replace('%)', '')
                    binding_info_dic_especifico["percent_2"] = line_segments[13].replace('(', '').replace('%)', '')
                if 'Query:' in line:
                    q_seq = line_segments[2]
                    binding_info_dic_especifico["q_seq"] = delete_no_interact_zone(q_seq)
                if 'Ref:' in line:
                    r_seq = line_segments[2]
                    binding_info_dic_especifico["r_seq"] = delete_no_interact_zone(r_seq)
                if 'Energy:' in line:
                    binding_info_dic_especifico["align_energy"] = line_segments[1]
                if 'No Hits Found above Threshold' in line:
                    negative = True
                    tamano_no_hits_found+=1


        if not negative:
            interior_dic, hairpin_dic, multi_dic, total_energy = get_binding_loops(binding_info_dic_especifico, lnc, mir)
            features = get_features(interior_dic, hairpin_dic, multi_dic, binding_info_dic_especifico)
            output.write(f'{lnc}_{mir},{",".join(features)},{total_energy}\n')
        elif negative:
            output.write(f'{lnc}_{mir},-1,-1,-1,0,0,0\n')
    print(tamano_no_hits_found)

100%|██████████| 45977/45977 [00:12<00:00, 3714.66it/s]

18012
